# VeriMail AI — Data Exploration & Feature Engineering

Builds the training dataset for phishing email detection by merging 6 public
email corpora (CEAS_08, Enron, Ling, Nazario, Nigerian_Fraud, SpamAssassin),
cleaning the text, and engineering interpretable signal features (urgency
language, credential-bait language, sender domain patterns, etc.) that power
both the ML models and the human-readable rating score shown in the app.

**Note on `phishing_email.csv`:** the raw data folder also contains a
pre-merged `phishing_email.csv`. We deliberately do NOT use it — it only has
`text_combined` + `label`, with no sender/URL columns, which would prevent
us from building the sender-domain and URL-based features below.

## 1. Load and standardize the 6 source datasets

In [3]:
import pandas as pd
import os

data_dir = "../data/raw"

def load_and_standardize(filename, source_name):
    """Load one source CSV and reduce it to a common schema.
    Sources vary in which columns they have (e.g. Enron/Ling lack
    sender/urls) — missing columns are added as NaN so every source
    concatenates cleanly."""
    df = pd.read_csv(os.path.join(data_dir, filename))

    for col in ['sender', 'subject', 'body', 'urls', 'label']:
        if col not in df.columns:
            df[col] = pd.NA

    df = df[['sender', 'subject', 'body', 'urls', 'label']].copy()
    df['source'] = source_name  # keep origin for later leakage checks
    return df

files_map = {
    "CEAS_08.csv": "CEAS_08",
    "Enron.csv": "Enron",
    "Ling.csv": "Ling",
    "Nazario.csv": "Nazario",
    "Nigerian_Fraud.csv": "Nigerian_Fraud",
    "SpamAssasin.csv": "SpamAssasin",
}

dfs = []
for fname, source in files_map.items():
    df_part = load_and_standardize(fname, source)
    print(f"{source}: {df_part.shape[0]} rows, label counts: {df_part['label'].value_counts().to_dict()}")
    dfs.append(df_part)

combined = pd.concat(dfs, ignore_index=True)
print(f"\nTotal combined: {combined.shape[0]} rows")
print(f"Overall label distribution:\n{combined['label'].value_counts()}")
print(f"\nMissing values per column:\n{combined.isna().sum()}")

CEAS_08: 39154 rows, label counts: {1: 21842, 0: 17312}
Enron: 29767 rows, label counts: {0: 15791, 1: 13976}
Ling: 2859 rows, label counts: {0: 2401, 1: 458}
Nazario: 1565 rows, label counts: {1: 1565}
Nigerian_Fraud: 3332 rows, label counts: {1: 3332}
SpamAssasin: 5809 rows, label counts: {0: 4091, 1: 1718}

Total combined: 82486 rows
Overall label distribution:
label
1    42891
0    39595
Name: count, dtype: int64

Missing values per column:
sender     32957
subject      347
body           1
urls       32626
label          0
source         0
dtype: int64


**Leakage risk found:** missing `sender`/`urls` values line up almost
exactly with which source file a row came from (Enron + Ling have no
sender/url data at all), not with the phishing/legitimate label. Left
unchecked, a model could learn "no sender field → legitimate" as a shortcut
instead of learning real phishing language. We handle this below by keeping
explicit `has_sender_info`/`has_url_info` flags and later stratifying the
train/test split by `source` as well as `label`.

## 2. Clean text and handle missing values

In [4]:
import re

# Drop the single row with no body at all — unusable
combined = combined.dropna(subset=['body']).reset_index(drop=True)

# Missing subject is itself a mild signal, not noise — keep as empty string
combined['subject'] = combined['subject'].fillna('')

# Leakage-aware flags: record whether metadata was present BEFORE filling it,
# so the model sees this explicitly rather than absorbing it as a hidden bias
combined['has_sender_info'] = combined['sender'].notna().astype(int)
combined['has_url_info'] = combined['urls'].notna().astype(int)

combined['sender'] = combined['sender'].fillna('unknown')
combined['urls'] = combined['urls'].fillna(0).astype(int)

def clean_text(text):
    """Lowercase, strip HTML/punctuation, normalize URLs and email
    addresses to placeholder tokens (keeps 'this had a link' as a
    frequency signal without letting noisy raw URLs into the vocabulary)."""
    text = str(text).lower()
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'http\S+|www\.\S+', ' URLPLACEHOLDER ', text)
    text = re.sub(r'\S+@\S+', ' EMAILPLACEHOLDER ', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

combined['subject_clean'] = combined['subject'].apply(clean_text)
combined['body_clean'] = combined['body'].apply(clean_text)

print(f"Final shape: {combined.shape}")
print(f"\nLabel distribution:\n{combined['label'].value_counts()}")
print(f"\nhas_sender_info vs label (leakage check):\n{pd.crosstab(combined['has_sender_info'], combined['label'])}")

Final shape: (82485, 10)

Label distribution:
label
1    42890
0    39595
Name: count, dtype: int64

has_sender_info vs label (leakage check):
label                0      1
has_sender_info              
0                18192  14765
1                21403  28125


## 3. Save the cleaned dataset

In [5]:
output_path = "../data/processed/combined_emails_clean.csv"
combined.to_csv(output_path, index=False)

# Reload check — catches CSV corruption from stray characters in raw text
check = pd.read_csv(output_path)
assert check.shape == combined.shape, "Row count mismatch after save/reload!"
print(f"Saved and verified {combined.shape[0]} rows to {output_path}")

Saved and verified 82485 rows to ../data/processed/combined_emails_clean.csv


## 4. Feature engineering — signal categories

Four categories of interpretable, human-explainable signals, chosen to power
both the ML models and the "flagged reasons" shown in the app UI:

1. **Urgency/pressure language** — "act now", "account suspended", etc.
2. **Financial/credential bait** — "verify your account", "click here", etc.
3. **Structural red flags** — caps ratio, exclamation marks, URL presence
4. **Generic greeting** — "Dear Customer" instead of a real name

In [6]:
df = pd.read_csv("../data/processed/combined_emails_clean.csv")

URGENCY_WORDS = [
    'urgent', 'immediately', 'act now', 'verify now', 'expire', 'expires',
    'expiring', 'suspend', 'suspended', 'action required', 'limited time',
    'deadline', 'final notice', 'warning', 'alert', 'restricted',
    'unauthorized', 'immediate action', 'respond immediately'
]

CREDENTIAL_BAIT_WORDS = [
    'password', 'ssn', 'social security', 'bank account', 'credit card',
    'click here', 'verify your account', 'confirm your identity',
    'update your information', 'login', 'log in', 'sign in',
    'account suspended', 'claim your', 'winner', 'prize', 'free gift',
    'wire transfer', 'billing', 'payment failed'
]

GENERIC_GREETINGS = [
    'dear customer', 'dear user', 'dear member', 'dear valued customer',
    'dear sir', 'dear madam', 'dear account holder', 'dear friend'
]

def count_matches(text, word_list):
    text = str(text).lower()
    return sum(text.count(w) for w in word_list)

def caps_ratio(text):
    text = str(text)
    return sum(1 for c in text if c.isupper()) / len(text) if len(text) else 0.0

def exclamation_count(text):
    return str(text).count('!')

def has_generic_greeting(text):
    text = str(text).lower()
    return int(any(g in text for g in GENERIC_GREETINGS))

def extract_sender_domain(sender):
    if pd.isna(sender) or sender == 'unknown':
        return 'unknown'
    match = re.search(r'@([\w\.-]+)', str(sender))
    return match.group(1).lower() if match else 'unknown'

# Use ORIGINAL (uncleaned) subject/body for caps/punctuation — cleaning
# lowercased everything, which would destroy the caps-ratio signal
df['urgency_score'] = df['body'].apply(lambda x: count_matches(x, URGENCY_WORDS)) + \
                       df['subject'].apply(lambda x: count_matches(x, URGENCY_WORDS))
df['credential_bait_score'] = df['body'].apply(lambda x: count_matches(x, CREDENTIAL_BAIT_WORDS)) + \
                               df['subject'].apply(lambda x: count_matches(x, CREDENTIAL_BAIT_WORDS))
df['caps_ratio_subject'] = df['subject'].apply(caps_ratio)
df['exclamation_count'] = df['subject'].apply(exclamation_count) + df['body'].apply(exclamation_count)
df['url_count'] = df['urls']  # 0/1 flag from source data
df['has_generic_greeting'] = df['body'].apply(has_generic_greeting)
df['sender_domain'] = df['sender'].apply(extract_sender_domain)

feature_cols = ['urgency_score', 'credential_bait_score', 'caps_ratio_subject',
                 'exclamation_count', 'url_count', 'has_generic_greeting']
print(df.groupby('label')[feature_cols].mean())

       urgency_score  credential_bait_score  caps_ratio_subject  \
label                                                             
0           0.304432               0.167875            0.065162   
1           0.510702               0.376661            0.114799   

       exclamation_count  url_count  has_generic_greeting  
label                                                      
0               1.054275   0.377548              0.001945  
1               1.550874   0.439636              0.050898  


**Result:** every feature moves in the expected direction. `has_generic_greeting`
is the standout — phishing emails are ~26x more likely to open with "Dear
Customer" than legitimate ones. `url_count` is the weakest (small gap), likely
because it's a coarse 0/1 flag rather than an actual link count.

### Sender domain check — is there a spoofing/free-webmail pattern?

In [7]:
print(df.groupby('label')['sender_domain'].apply(lambda x: x.value_counts().head(5)))

label                        
0      unknown                   18192
       gmail.com                  2614
       spamassassin.taint.org      641
       python.org                  556
       issues.apache.org           462
1      unknown                   15752
       hotmail.com                 519
       yahoo.com                   492
       google.com                  209
       virgilio.it                 159
Name: sender_domain, dtype: int64


**Finding:** legitimate top domains are corporate/infrastructure
(`gmail.com`, `python.org`, `issues.apache.org`); phishing top domains are
overwhelmingly free consumer webmail (`hotmail.com`, `yahoo.com`,
`virgilio.it`). Matches the real-world pattern of attackers using free,
disposable accounts. Not a clean binary signal though — `gmail.com` appears
heavily in both classes — so this becomes a *mild* contributing feature
(`is_free_webmail`), not a dominant one.

### Check for raw HTML in body (would enable link-text-vs-href mismatch features)

In [8]:
html_count = df['body'].str.contains('<html|<body|href=', case=False, na=False).sum()
print(f"{html_count} of {len(df)} rows contain HTML markup in body")

39 of 82485 rows contain HTML markup in body


Only ~0.05% of rows have raw HTML — bodies are essentially plain-text
extracted across the dataset, so an href-vs-display-text mismatch feature
isn't worth building here.

## 5. Additional raw features: brand/domain mismatch, suspicious TLD, word count

In [9]:
BRAND_DOMAINS = {
    'paypal': 'paypal.com', 'amazon': 'amazon.com', 'apple': 'apple.com',
    'microsoft': 'microsoft.com', 'google': 'google.com', 'netflix': 'netflix.com',
    'bank of america': 'bankofamerica.com', 'chase': 'chase.com', 'wells fargo': 'wellsfargo.com',
    'ebay': 'ebay.com', 'facebook': 'facebook.com', 'instagram': 'instagram.com',
    'irs': 'irs.gov', 'fedex': 'fedex.com', 'ups': 'ups.com', 'dhl': 'dhl.com'
}

def brand_domain_mismatch(row):
    text = (str(row['subject']) + ' ' + str(row['body'])).lower()
    domain = row['sender_domain']
    for brand, real_domain in BRAND_DOMAINS.items():
        if brand in text and domain != 'unknown' and real_domain not in domain:
            return 1
    return 0

SUSPICIOUS_TLDS = ['.tk', '.ml', '.ga', '.cf', '.gq', '.xyz', '.top', '.click', '.work', '.link']

def has_suspicious_tld(domain):
    return int(domain != 'unknown' and any(domain.endswith(tld) for tld in SUSPICIOUS_TLDS))

df['brand_domain_mismatch'] = df.apply(brand_domain_mismatch, axis=1)
df['suspicious_tld'] = df['sender_domain'].apply(has_suspicious_tld)
df['word_count'] = df['body'].apply(lambda x: len(str(x).split()))

new_features = ['brand_domain_mismatch', 'suspicious_tld', 'word_count']
print(df.groupby('label')[new_features].mean())

       brand_domain_mismatch  suspicious_tld  word_count
label                                                   
0                   0.168759        0.000000  356.965450
1                   0.104570        0.000816  195.405526


**Findings — two out of three are worth keeping, one is not:**

- `word_count`: strong, expected signal — phishing averages ~195 words vs
  ~357 for legitimate (roughly half the length). **Keep.**
- `suspicious_tld`: directionally correct but only fires on ~67 emails total
  in this dataset — too rare to move metrics much on this corpus, but cheap
  to keep for real-world traffic where sketchy TLDs are more common. **Keep.**
- `brand_domain_mismatch`: **inverted** — legitimate emails trigger it more
  than phishing. Root cause: Enron dominates the legitimate class, and
  ordinary internal business email mentions companies by name
  ("the Chase deal", "the Amazon invoice") from an `enron.com` address,
  which this simple check misreads as a mismatch. A feature that points the
  wrong direction would actively mislead the score if weighted positively.
  **Excluded from the final rating score** — kept in the code above only to
  document that it was tested and why it was dropped.

## 6. Derived features: free webmail provider, short message body

In [10]:
FREE_WEBMAIL_DOMAINS = [
    'gmail.com', 'yahoo.com', 'hotmail.com', 'outlook.com', 'aol.com',
    'virgilio.it', 'live.com', 'mail.com', 'protonmail.com', 'gmx.com',
    'yandex.com', 'icloud.com'
]
df['is_free_webmail'] = df['sender_domain'].apply(lambda d: int(d in FREE_WEBMAIL_DOMAINS))

# word_count is continuous and inversely related to phishing (shorter = more
# suspicious) — convert to boolean so it fits the weighted-sum scoring below
df['word_count_short'] = (df['word_count'] < 150).astype(int)

print(df.columns.tolist())

['sender', 'subject', 'body', 'urls', 'label', 'source', 'has_sender_info', 'has_url_info', 'subject_clean', 'body_clean', 'urgency_score', 'credential_bait_score', 'caps_ratio_subject', 'exclamation_count', 'url_count', 'has_generic_greeting', 'sender_domain', 'brand_domain_mismatch', 'suspicious_tld', 'word_count', 'is_free_webmail', 'word_count_short']


## 7. Weighted rating score + flagged reasons

Final feature set: 9 signals (the original 6 + `suspicious_tld` +
`is_free_webmail` + `word_count_short`), `brand_domain_mismatch` excluded per
the finding above. Weights are set roughly proportional to the effect size
seen in the validation checks above — `has_generic_greeting` and
`credential_bait_score` get the most weight, `url_count` and `is_free_webmail`
the least.

This rule-based score is deliberately separate from the ML model trained in
`01_model_training.ipynb` — it's instant (no model inference needed) and
self-explaining ("here's exactly why this scored 72/100"), which a raw
classifier probability can't offer on its own. The app shows both.

In [11]:
WEIGHTS = {
    'urgency_score': 8,
    'credential_bait_score': 12,
    'caps_ratio_subject': 15,
    'exclamation_count': 3,
    'url_count': 5,
    'has_generic_greeting': 25,
    'is_free_webmail': 6,
    'suspicious_tld': 10,
    'word_count_short': 6,
}

MAX_RAW_SCORE = (
    WEIGHTS['urgency_score'] * 5 +
    WEIGHTS['credential_bait_score'] * 5 +
    WEIGHTS['caps_ratio_subject'] * 1 +
    WEIGHTS['exclamation_count'] * 5 +
    WEIGHTS['url_count'] * 1 +
    WEIGHTS['has_generic_greeting'] * 1 +
    WEIGHTS['is_free_webmail'] * 1 +
    WEIGHTS['suspicious_tld'] * 1 +
    WEIGHTS['word_count_short'] * 1
)

def compute_rating_score(row):
    raw = (
        min(row['urgency_score'], 5) * WEIGHTS['urgency_score'] +
        min(row['credential_bait_score'], 5) * WEIGHTS['credential_bait_score'] +
        row['caps_ratio_subject'] * WEIGHTS['caps_ratio_subject'] +
        min(row['exclamation_count'], 5) * WEIGHTS['exclamation_count'] +
        min(row['url_count'], 1) * WEIGHTS['url_count'] +
        row['has_generic_greeting'] * WEIGHTS['has_generic_greeting'] +
        row['is_free_webmail'] * WEIGHTS['is_free_webmail'] +
        row['suspicious_tld'] * WEIGHTS['suspicious_tld'] +
        row['word_count_short'] * WEIGHTS['word_count_short']
    )
    return min(100, round((raw / MAX_RAW_SCORE) * 100, 1))

def get_flagged_reasons(row):
    reasons = []
    if row['urgency_score'] > 0:
        reasons.append(f"Urgency language detected ({int(row['urgency_score'])} instance(s))")
    if row['credential_bait_score'] > 0:
        reasons.append(f"Credential/financial bait language detected ({int(row['credential_bait_score'])} instance(s))")
    if row['caps_ratio_subject'] > 0.3:
        reasons.append("Excessive capitalization in subject line")
    if row['exclamation_count'] >= 3:
        reasons.append(f"Excessive exclamation marks ({int(row['exclamation_count'])})")
    if row['url_count'] > 0:
        reasons.append("Contains embedded URL(s)")
    if row['has_generic_greeting'] == 1:
        reasons.append("Generic greeting (e.g. 'Dear Customer') instead of personalized name")
    if row['is_free_webmail'] == 1:
        reasons.append(f"Sent from free webmail provider ({row['sender_domain']})")
    if row['suspicious_tld'] == 1:
        reasons.append(f"Sender domain uses a high-risk TLD ({row['sender_domain']})")
    if row['word_count_short'] == 1:
        reasons.append("Unusually short message body")
    return reasons

df['rating_score'] = df.apply(compute_rating_score, axis=1)
df['flagged_reasons'] = df.apply(get_flagged_reasons, axis=1)

print(df.groupby('label')['rating_score'].describe())

         count       mean        std  min  25%  50%   75%   max
label                                                          
0      39595.0   6.049547   6.672188  0.0  3.2  4.1   7.6  73.1
1      42890.0  10.802837  10.380813  0.0  4.8  7.2  11.8  86.2


**Result:** phishing averages a rating score of ~10.8 vs ~6.0 for
legitimate — roughly 1.8x higher, and the gap holds across every percentile,
not just the mean. Scores stay well under 100 even for the worst offenders
(max ~86) because `MAX_RAW_SCORE` assumes every signal fires at once, which
real emails rarely do — this gives the score real headroom rather than
everything clustering near the ceiling.

## 8. Save the final feature-engineered dataset

In [12]:
output_path = "../data/processed/emails_features_final.csv"
df.to_csv(output_path, index=False)
print(f"Saved {df.shape[0]} rows, {df.shape[1]} columns to {output_path}")

Saved 82485 rows, 24 columns to ../data/processed/emails_features_final.csv
